# PRJEB44932 - Jahn et al.

Article link: []().

In [1]:
suppressPackageStartupMessages({
    library(provoc)
    library(here)
    library(ggplot2)
    library(lubridate)
})

prj <- "PRJEB44932"
lineages_in_paper <- c("B.1.351", "P.1", "B.1.1.7", "B.1.617.2")


In [2]:
coco <- read.csv(
    here(
        "data/processed/",
        paste0(prj, "_processed.csv.gz")
    )
)
coco$mutation <- parse_mutations(coco$label)
head(coco)


  position    label    mutation frequency coverage count        sra       date  sample_name avg_spot_len      bases bioproject location     lat    lon
1    10046 +10047.T ins:10047:1         0     2982     0 ERR5922163 2021-03-09 SAMEA8745766          502 9782804316 PRJEB44932 Lausanne 46.5202 6.5929
2    10221 +10222.T ins:10222:1         0      986     0 ERR5922163 2021-03-09 SAMEA8745766          502 9782804316 PRJEB44932 Lausanne 46.5202 6.5929
3    10360 +10361.T ins:10361:1         0      987     0 ERR5922163 2021-03-09 SAMEA8745766          502 9782804316 PRJEB44932 Lausanne 46.5202 6.5929
4     1051  +1052.T  ins:1052:1         0    72150     0 ERR5922163 2021-03-09 SAMEA8745766          502 9782804316 PRJEB44932 Lausanne 46.5202 6.5929
5    10573 +10574.A ins:10574:1         0     6230     0 ERR5922163 2021-03-09 SAMEA8745766          502 9782804316 PRJEB44932 Lausanne 46.5202 6.5929
6    10712 +10713.T ins:10713:1         0   497888     0 ERR5922163 2021-03-09 SAMEA8745766   

In [3]:
barcodes <- provoc::usher_barcodes()
print("Lineages not in barcodes:")
print(setdiff(lineages_in_paper, rownames(barcodes)))
available_lineages <- intersect(lineages_in_paper, rownames(barcodes))
barcodes <- filter_lineages(barcodes, available_lineages)
dim(barcodes)


[1] "Lineages not in barcodes:"
character(0)


[1]  4 68

In [ ]:
system.time(
    res <- provoc(count / coverage ~ .,
        data = coco,
        lineage_defs = barcodes,
        by = "sra")
)
head(res)


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12)

res$date <- lubridate::ymd(res$date)

gg <- autoplot(res, date_col = "date") + facet_wrap(~location) +
    geom_smooth(formula = y ~ x, se = FALSE, method = "loess") +
    theme_bw()
suppressWarnings(print(gg))
